# script to compute the feasible output mapping for example 02


In [2]:
import numpy as np
import scipy.linalg

# ==========================================
# 1. System parameters
# ==========================================
L = 50.0  # rocket length (m)
g = 9.8  # gravitational acceleration (m/s^2)
# Final mass at hover: initial 100000 kg, dry mass 85000 kg.
# We linearise at the dry mass (end-of-burn condition).
m_f = 85000.0

# ==========================================
# 2. Linearised system matrices A and B
# ==========================================
# State vector: X = [x, y, theta, vx, vy, omega]^T
# Dynamics linearised around X_eq = 0, U_eq = [0, m_f*g]^T
A = np.zeros((6, 6))
A[0, 3] = 1.0  # dx/dt = vx
A[1, 4] = 1.0  # dy/dt = vy
A[2, 5] = 1.0  # dtheta/dt = omega
A[3, 2] = g  # dvx/dt: coupling of thrust and gravity through tilt angle theta

# Input vector: U = [u1, u2]^T
# u1: lateral thrust component (T*sin(delta))
# u2: axial thrust increment   (T*cos(delta) - m_f*g)
B = np.zeros((6, 2))
B[3, 0] = 1.0 / m_f
B[4, 1] = 1.0 / m_f
B[5, 0] = -6.0 / (m_f * L)

# ==========================================
# 3. LQR design to obtain the output matrix C
# ==========================================
# Q: state penalty — position and angle are penalised more heavily than velocities
Q = np.diag([100.0, 100.0, 500.0, 10.0, 10.0, 50.0])

# R: control penalty — thrust inputs are in Newtons (order of 10^5),
# so R is set small to avoid numerical issues (equivalent to input normalisation)
R = np.diag([1e-6, 1e-6])

# Solve the continuous-time algebraic Riccati equation
P = scipy.linalg.solve_continuous_are(A, B, Q, R)

# Compute the LQR feedback gain K (used as the virtual output matrix C)
K = np.linalg.inv(R) @ B.T @ P
C = K

# ==========================================
# 4. Print results
# ==========================================
np.set_printoptions(precision=4, suppress=True)
print("=== Linearised system matrix A (6x6) ===")
print(A)
print("\n=== Input matrix B (6x2) ===")
print(B)
print("\n=== Virtual output matrix C (2x6) ===")
print(C.T)

=== Linearised system matrix A (6x6) ===
[[0.  0.  0.  1.  0.  0. ]
 [0.  0.  0.  0.  1.  0. ]
 [0.  0.  0.  0.  0.  1. ]
 [0.  0.  9.8 0.  0.  0. ]
 [0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0. ]]

=== Input matrix B (6x2) ===
[[ 0.  0.]
 [ 0.  0.]
 [ 0.  0.]
 [ 0.  0.]
 [ 0.  0.]
 [-0.  0.]]

=== Virtual output matrix C (2x6) ===
[[  -10000.           -0.    ]
 [       0.        10000.    ]
 [-1050364.3266       -0.    ]
 [  -46406.9032       -0.    ]
 [       0.        41352.1463]
 [-1557495.3147       -0.    ]]
